# 05. 2022년 이후 청년 고용률 — 무슨 일이 있었고, 앞으로 어떻게 될까

연구 질문(설계 문서 8-12절): 생성형 AI 출시(2022.11) 이후 청년(**20~29세**) 고용률은 얼마나 변했고, AI 성능(ECI)이 계속 오르면 앞으로 어떻게 될까?

- 입력: `data/raw/kosis/DT_1DA7002S_*.csv`(월별 전국), `data/processed/panel_analysis.csv`(직업별 반기), `data/raw/epoch_eci/eci_scores.csv`
- 범위: **2022년 1월 이후만** 쓴다. 원본 파일은 그대로 두고 이 노트북에서만 자른다.
- 출력: `reports/post2022_results.csv`, `reports/figures/fig12~14`

**연령 기준**: 전국 고용률(1·3-1·3-2·4절)은 20~29세로 본다. 15~19세는 대부분 학생이라 고용률이 8% 안팎이고, 이들의 인구 비중이 바뀌기만 해도 15~29세 고용률이 움직이기 때문이다(2022년 이후 15~29세 고용률 하락의 약 1%p가 이 구성 변화였다). 직업별 표(2·3-3절)는 15~29세 구간만 제공되므로 그대로 쓴다. 청년 취업자 중 15~18세는 약 1%라 영향이 작다.

2022년 이전을 보지 않으므로, 이 노트북의 숫자는 "2022년 이후 함께 움직인 정도"이지 "AI가 대체한 양"이 아니다.
노출 직업의 청년 감소가 AI 이전에도 있었다는 점은 03·04 노트북(설계 문서 8-10·8-11절)에 있다.

In [ ]:
import sys
import warnings

sys.path.append("..")  # 노트북 폴더 기준으로 src를 찾는다 (README 6절)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from matplotlib import font_manager

from src import config

warnings.filterwarnings("ignore", category=FutureWarning)
INK2, MUTED, GRID, AXIS, SURFACE = "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
BLUE, ORANGE, TERC = "#2a78d6", "#eb6834", {"하": "#86b6ef", "중": "#2a78d6", "상": "#0d366b"}
_fonts = {f.name for f in font_manager.fontManager.ttflist}
FONT = next(f for f in ["AppleGothic", "Malgun Gothic", "NanumGothic", "DejaVu Sans"] if f in _fonts)
sns.set_theme(style="whitegrid", font=FONT, rc={
    "axes.unicode_minus": False, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.edgecolor": AXIS, "axes.labelcolor": INK2,
    "xtick.color": MUTED, "ytick.color": MUTED, "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.labelsize": 9,
    "axes.titleweight": "bold", "axes.titlesize": 10, "axes.titlelocation": "left",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})
START = "2022-01-01"
rows = []   # 결과 표에 쌓을 숫자

## 0. 자료 읽기

월별 표는 KOSIS 원본 그대로다(1행 = 시점, 2행 = 항목). 고용률 = 취업자 ÷ 인구 × 100.
ECI는 매달 15일(조사 기준일)까지 나온 모델 중 최고점이다.

In [ ]:
raw = pd.read_csv(next((config.DATA_RAW / "kosis").glob("DT_1DA7002S_*.csv")), encoding="cp949", header=None, dtype=str)
per, item = raw.iloc[0, 1:].str[1:7], raw.iloc[1, 1:].str[:3]


def series(age, it):
    r = raw[raw[0].str.startswith(age)].iloc[0, 1:]
    s = pd.to_numeric(r[(item == it).values], errors="coerce")
    s.index = pd.to_datetime(per[(item == it).values].values, format="%Y%m")
    return s


m = pd.DataFrame({"pop": series("20 ", "T10"), "emp": series("20 ", "T30")})   # "20 " = 20~29세
m["er"] = 100 * m.emp / m["pop"]
m["er_a"] = 100 * sum(series(a, "T30") for a in ["30 ", "40 ", "50 "]) / sum(series(a, "T10") for a in ["30 ", "40 ", "50 "])
eci_m = pd.read_csv(config.DATA_RAW / "epoch_eci" / "eci_scores.csv", parse_dates=["date"])
m["eci"] = [eci_m[eci_m.date <= t + pd.Timedelta(days=14)].eci.max() for t in m.index]
m = m[m.index >= START]
assert m[["pop", "emp", "er", "er_a"]].notna().all().all()
print(f"월별 자료 {m.index.min():%Y.%m} ~ {m.index.max():%Y.%m}, {len(m)}개월")

## 1. 무슨 일이 있었나 — 청년 고용률과 취업자 수

2022년 평균과 최근 12개월 평균을 비교한다(계절 차이를 없애려고 12개월씩 묶음).
취업자 변화 = 인구가 변한 몫(고용률은 2022년 그대로라고 가정) + 고용률이 변한 몫.

In [ ]:
y0 = m[m.index.year == 2022].mean()
y1 = m.iloc[-12:].mean()
last12 = f"{m.index[-12]:%Y.%m}~{m.index[-1]:%Y.%m}"
r0, r1 = y0.emp / y0["pop"], y1.emp / y1["pop"]   # 평균 취업자 ÷ 평균 인구 (월별 고용률의 평균과 0.01%p 안팎 차이)
d_pop = (y1["pop"] - y0["pop"]) * r0
d_rate = y1["pop"] * (r1 - r0)
assert abs(d_pop + d_rate - (y1.emp - y0.emp)) < 1e-6
summary = pd.DataFrame({"2022년 평균": y0, f"최근 12개월({last12})": y1}).loc[["er", "er_a", "pop", "emp", "eci"]]
summary.index = ["20~29세 고용률(%)", "30~59세 고용률(%)", "20~29세 인구(천명)", "20~29세 취업자(천명)", "ECI(최고점)"]
summary["변화"] = summary.iloc[:, 1] - summary.iloc[:, 0]
print(summary.round(2).to_string())
print(f"\n20~29세 취업자 변화 {y1.emp - y0.emp:+.0f}천명 = 인구 감소 몫 {d_pop:+.0f} + 고용률 하락 몫 {d_rate:+.0f}")
rows += [{"구분": "1. 변화", "항목": k, "값": v} for k, v in
         {"20~29세 고용률 변화(%p)": y1.er - y0.er, "30~59세 고용률 변화(%p)": y1.er_a - y0.er_a, "20~29세 취업자 변화(천명)": y1.emp - y0.emp,
          "그중 인구 감소 몫(천명)": d_pop, "그중 고용률 하락 몫(천명)": d_rate}.items()]
print("\n연평균 20~29세 고용률(%):", m.er.groupby(m.index.year).mean().round(2).to_dict())

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
roll = m.er.rolling(12).mean()
a1.plot(m.index, m.er, color=BLUE, lw=1, alpha=0.35, label="월별")
a1.plot(roll.index, roll, color=BLUE, lw=2.2, label="최근 12개월 평균")
a1.axvline(pd.Timestamp("2022-11-30"), color=MUTED, ls="--", lw=1)
a1.text(pd.Timestamp("2022-12-15"), 0.97, "ChatGPT 출시", transform=a1.get_xaxis_transform(), color=MUTED, fontsize=8, va="top")
a1.set_ylabel("20~29세 고용률 %")
a1.legend(frameon=False, fontsize=8, loc="lower left")
a1.set_title("20~29세 고용률은 2024년 이후 내려가고 있다")
a2.step(m.index, m.eci, where="post", color=ORANGE, lw=2)
a2.set_ylabel("ECI 최고점")
a2.set_title("같은 기간 AI 성능(ECI)", fontsize=9)
for ax in (a1, a2):
    ax.grid(axis="x", visible=False)
fig.text(0.01, -0.02, "자료: KOSIS 연령별 경제활동인구(월), Epoch AI ECI(2026-11-11 파일). ECI는 2023년 2월부터 있음.", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig(config.FIGURES / "fig12_청년고용률_ECI_월별.png")
plt.show()

## 2. 어디서 줄었나 — AI 노출도별 직업

직업 중분류 56개를 AI 노출도로 3등분하고, 2022년 평균과 최근 두 반기(2025 하, 2026 상) 평균을 비교한다.
직업별 인구 자료가 없어서 고용률 대신 취업자 수를 본다. 직업별 표의 청년은 15~29세다(이 표에 다른 구간이 없음).

In [ ]:
p = pd.read_csv(config.DATA_PROCESSED / "panel_analysis.csv", dtype={"code": str, "period": str})
p = p[p.period >= "202201"]
before = p[p.year == 2022].groupby(["code", "name", "terc"])[["y15_29", "a30_59"]].mean()
after = p[p.period.isin(["202502", "202601"])].groupby(["code", "name", "terc"])[["y15_29", "a30_59"]].mean()
tb, ta = before.groupby("terc").sum().loc[["하", "중", "상"]], after.groupby("terc").sum().loc[["하", "중", "상"]]
terc = pd.DataFrame({"청년 변화(천명)": ta.y15_29 - tb.y15_29, "청년 변화(%)": 100 * (ta.y15_29 / tb.y15_29 - 1),
                     "30~59세 변화(천명)": ta.a30_59 - tb.a30_59, "30~59세 변화(%)": 100 * (ta.a30_59 / tb.a30_59 - 1),
                     "청년 감소 중 비중(%)": 100 * (ta.y15_29 - tb.y15_29) / (ta.y15_29 - tb.y15_29).sum(),
                     "2022년 청년 비중(%)": 100 * tb.y15_29 / tb.y15_29.sum()})
print(terc.round(1).to_string())
for t in terc.index:
    rows.append({"구분": "2. 노출도별", "항목": f"노출도 {t}: 청년 변화(%)", "값": terc.loc[t, "청년 변화(%)"]})
    rows.append({"구분": "2. 노출도별", "항목": f"노출도 {t}: 30~59세 변화(%)", "값": terc.loc[t, "30~59세 변화(%)"]})
occ = (after - before).join(100 * (after / before - 1), rsuffix="_pct").reset_index().sort_values("y15_29")
print("\n청년 취업자가 가장 많이 준 직업 10개")
print(occ.head(10).round(1).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
xs = np.arange(3)
ax.bar(xs - 0.2, terc["청년 변화(%)"], 0.38, color=[TERC[t] for t in terc.index], label="청년(15~29세)")
ax.bar(xs + 0.2, terc["30~59세 변화(%)"], 0.38, color="white", edgecolor=[TERC[t] for t in terc.index], hatch="///", lw=1.2, label="30~59세")
for x, (yv, av) in enumerate(zip(terc["청년 변화(%)"], terc["30~59세 변화(%)"])):
    ax.text(x - 0.2, yv - 0.6, f"{yv:.1f}%", ha="center", va="top", fontsize=8, color=INK2)
    ax.text(x + 0.2, av + (0.5 if av >= 0 else -0.6), f"{av:+.1f}%", ha="center", va="bottom" if av >= 0 else "top", fontsize=8, color=INK2)
ax.axhline(0, color=AXIS, lw=1)
ax.set_xticks(xs)
ax.set_xticklabels([f"AI 노출도 {t}" for t in terc.index])
ax.set_ylabel("2022년 대비 취업자 변화 %")
ax.set_ylim(min(terc["청년 변화(%)"].min(), terc["30~59세 변화(%)"].min()) - 4, terc["30~59세 변화(%)"].max() + 4)
ax.legend(frameon=False, fontsize=8, loc="upper left")
ax.set_title("노출도 상위 직업에서는 30~59세가 늘 때 청년만 줄었다")
ax.grid(axis="x", visible=False)
fig.text(0.01, -0.03, "자료: KOSIS 지역별고용조사 직업 중분류 56개. 2022년 평균 대비 2025 하~2026 상 평균.", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig(config.FIGURES / "fig13_노출도별_취업자_변화.png")
plt.show()

## 3. AI 성능과 함께 움직였나

### 3-1. 전국: ECI 10점당 청년 고용률 변화

계절 차이를 없애려고 "전년 같은 달 대비 변화"끼리 비교한다(2023년 3월부터 ECI가 있으므로 2024년 3월부터 계산 가능).
이웃한 달끼리 비슷한 점을 감안해 표준오차를 넓게 잡는다(HAC, 12개월).

In [ ]:
m["d_er"], m["d_er_a"], m["d_eci"] = m.er.diff(12), m.er_a.diff(12), m.eci.diff(12) / 10
q = m.dropna(subset=["d_er", "d_eci"])
nat = {}
for col, name in [("d_er", "20~29세"), ("d_er_a", "30~59세")]:
    r = sm.OLS(q[col], sm.add_constant(q.d_eci)).fit(cov_type="HAC", cov_kwds={"maxlags": 12})
    lo, hi = r.conf_int().loc["d_eci"]
    nat[name] = (r.params["d_eci"], lo, hi)
    print(f"{name} 고용률: ECI 10점당 {r.params['d_eci']:+.2f}%p (95% {lo:+.2f} ~ {hi:+.2f}), p = {r.pvalues['d_eci']:.3f}, {len(q)}개월")
    rows.append({"구분": "3. ECI 관계(전국)", "항목": f"{name} 고용률, ECI 10점당(%p)", "값": r.params["d_eci"], "하한": lo, "상한": hi})
print("ECI와 시간의 상관:", round(np.corrcoef(q.eci, np.arange(len(q)))[0, 1], 3))

### 3-2. 예측이 맞는지 — 학습에 쓰지 않은 기간으로 확인

2025년 6월까지로 모형을 맞추고, 그 뒤 달의 청년 고용률을 맞혀 본다. 비교 대상은 "시간이 지나면 일정하게 떨어진다"는 모형과
"작년 같은 달과 같다"는 가정이다. 셋 다 월별 고정효과(계절)를 넣는다.

In [ ]:
s = m[m.eci.notna()].copy()
s["t"] = np.arange(len(s)) / 12
M = pd.get_dummies(s.index.month, prefix="m", drop_first=True).astype(float).set_axis(s.index)
train = s.index <= "2025-06-01"
bt = {}
for name, cols in [("ECI 모형", ["eci"]), ("시간 추세 모형", ["t"])]:
    X = sm.add_constant(pd.concat([s[cols], M], axis=1))
    bt[name] = sm.OLS(s.er[train], X[train]).fit().predict(X[~train])
bt["작년 같은 달과 같음"] = m.er.shift(12).reindex(s.index[~train])
err = pd.DataFrame({k: s.er[~train] - v for k, v in bt.items()})
backtest = pd.DataFrame({"평균 오차(%p)": err.mean(), "평균 절대 오차(%p)": err.abs().mean()})
print(f"검증 기간 {s.index[~train].min():%Y.%m} ~ {s.index[~train].max():%Y.%m} ({(~train).sum()}개월)")
print(backtest.round(2).to_string())
rows += [{"구분": "3. 예측 검증", "항목": f"{k} 평균 절대 오차(%p)", "값": v} for k, v in backtest["평균 절대 오차(%p)"].items()]

### 3-3. 직업별: 노출도 높은 직업에서 청년이 더 줄었나

2022년(상·하반기)을 AI 이전 기준으로 두고 2023 상 ~ 2026 상과 비교한다. 결과변수는 ln(청년 ÷ 30~59세), 직업·시점 고정효과,
2022년 청년 취업자 가중, 직업 단위로 묶은 표준오차. 주 분석 표본 50개 직업.

In [ ]:
o = p[p.in_main & (p.w > 0)].copy()
o["eci_gain"] = ((o.eci - o.loc[o.period == "202301", "eci"].iloc[0]) / 10).fillna(0)
o["x_post"], o["x_eci"] = o.exp_z * o.post, o.exp_z * o.eci_gain


def fit(term):
    X = sm.add_constant(pd.concat([o[[term]], pd.get_dummies(o.period, prefix="p", drop_first=True),
                                   pd.get_dummies(o.code, prefix="c", drop_first=True)], axis=1).astype(float))
    return sm.WLS(o.lr_y, X, weights=o.w).fit(cov_type="cluster", cov_kwds={"groups": o.code.astype("category").cat.codes})


for term, label in [("x_post", "2023년 이후 평균"), ("x_eci", "ECI 10점당")]:
    r = fit(term)
    lo, hi = r.conf_int().loc[term] * 100
    print(f"노출도 1표준편차당 청년÷30~59세, {label}: {r.params[term] * 100:+.2f}% (95% {lo:+.2f} ~ {hi:+.2f}), p = {r.pvalues[term]:.3f}")
    rows.append({"구분": "3. ECI 관계(직업별)", "항목": f"노출도 1SD당 청년÷30~59세, {label}(%)", "값": r.params[term] * 100, "하한": lo, "상한": hi})
print(f"{o.code.nunique()}개 직업 × {o.period.nunique()}개 반기 = {len(o)}행")

## 4. 앞으로 — ECI가 더 오르면 (조건부 시나리오)

3-1의 관계(ECI 10점당 20~29세 고용률 변화)가 그대로 이어진다고 가정한다. 기준점은 최근 12개월의 평균 고용률과 평균 ECI다.
인원은 청년 인구를 최근 12개월 평균으로 고정해 계산한다(앞으로의 인구 감소는 반영하지 않음).
ECI가 언제 그 수준에 닿는지는 2023.03~최근 ECI 최고점의 직선 추세로 어림한다.

In [ ]:
b, lo, hi = nat["20~29세"]
er0, pop0, eci0 = y1.er, y1["pop"], y1.eci          # 기준점: 최근 12개월 평균
pace = np.polyfit(np.arange(len(s)) / 12, s.eci, 1)[0]
now = m.eci.iloc[-1]
scen = pd.DataFrame({"ECI": [now, 180, 190, 200]})
scen["시점"] = [f"{m.index[-1]:%Y.%m} (현재)"] + [f"~{(m.index[-1] + pd.DateOffset(months=round(12 * (e - now) / pace))):%Y.%m}" for e in scen.ECI[1:]]
gain = (scen.ECI - eci0) / 10
for k, c in [("중앙", b), ("덜 떨어지는 쪽", hi), ("더 떨어지는 쪽", lo)]:
    scen[f"고용률 {k}(%)"] = er0 + c * gain
scen["기준 대비 취업자 중앙(천명)"] = pop0 * b * gain / 100
print(f"기준점: 최근 12개월 20~29세 고용률 {er0:.2f}%, ECI 평균 {eci0:.1f}, 20~29세 인구 {pop0:,.0f}천명 / ECI 상승 속도 연 {pace:.1f}점")
print(scen.round(2).to_string(index=False))
for _, r in scen.iterrows():
    rows.append({"구분": "4. 시나리오", "항목": f"ECI {r.ECI:.0f}일 때 20~29세 고용률(%)", "값": r["고용률 중앙(%)"],
                 "하한": r["고용률 더 떨어지는 쪽(%)"], "상한": r["고용률 덜 떨어지는 쪽(%)"]})

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(m.eci.rolling(12).mean(), m.er.rolling(12).mean(), color=BLUE, lw=2, label="실제 (고용률·ECI 모두 12개월 평균)")
g = np.linspace(0, 200 - eci0, 50)
ax.fill_between(eci0 + g, er0 + lo * g / 10, er0 + hi * g / 10, color=ORANGE, alpha=0.15, lw=0, label="시나리오 95% 범위")
ax.plot(eci0 + g, er0 + b * g / 10, color=ORANGE, lw=2, ls="--", label="시나리오 중앙")
for _, r in scen.iloc[1:].iterrows():
    ax.annotate(f"ECI {r.ECI:.0f}: {r['고용률 중앙(%)']:.1f}%\n({r['시점']})", (r.ECI, r["고용률 중앙(%)"]),
                textcoords="offset points", xytext=(0, 10), ha="center", fontsize=7.5, color=INK2)
ax.set_xlim(118, 208)
ax.set_xlabel("ECI 최고점")
ax.set_ylabel("20~29세 고용률 %")
ax.legend(frameon=False, fontsize=8, loc="lower left")
ax.set_title("2022년 이후의 관계가 이어진다면: ECI 상승에 따른 20~29세 고용률")
fig.text(0.01, -0.03, "조건부 시나리오이며 예측이 아님. 범위는 ECI 10점당 계수의 95% 신뢰구간만 반영(경기·인구 변화 미반영).", color=MUTED, fontsize=8)
fig.tight_layout()
fig.savefig(config.FIGURES / "fig14_ECI_시나리오.png")
plt.show()

## 5. 저장

In [ ]:
res = pd.DataFrame(rows).round(3)
res.to_csv(config.FIGURES.parent / "post2022_results.csv", index=False, encoding="utf-8-sig")
print(res.to_string(index=False))